<a href="https://colab.research.google.com/github/mhovhannisyan06/handf/blob/main/TextGeminiRead.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ARMENIAN TEXT: how well does Gemini read handwritten Armenian?
from google.colab import drive
drive.mount('/content/drive')
import os, re, json, time, pandas as pd
from PIL import Image
from google import genai
from google.colab import userdata

BASE  = '/content/drive/MyDrive/handf_data'
LINES = os.path.join(BASE, 'text_lines')

# find the labels file (text_labels.csv or text.csv)
LABELS = next((os.path.join(BASE, n) for n in ['text_labels.csv', 'text.csv'] if os.path.exists(os.path.join(BASE, n))), None)
assert LABELS, 'No text_labels.csv / text.csv in handf_data'
assert os.path.isdir(LINES), 'Folder handf_data/text_lines is missing: upload it first!'
tl = pd.read_csv(LABELS, keep_default_na=False)
tl = tl[tl['text'].str.strip() != '']
print(len(tl), 'lines with your text,', len(os.listdir(LINES)), 'images in text_lines')

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
MODEL = 'gemini-3.5-flash-lite'
PROMPT = ('This image shows one line of handwritten Armenian text from math lecture notes. '
          'Transcribe it exactly as written, in Armenian script. Write math symbols as plain characters '
          '(for example: X=a, f(x,y), ∀, ∃). Output only the transcription, nothing else.')
CACHE = os.path.join(BASE, 'gemini_text_cache.json')
cache = json.load(open(CACHE)) if os.path.exists(CACHE) else {}

def ask(img):
    for attempt in range(5):
        try:
            return client.models.generate_content(model=MODEL, contents=[img, PROMPT]).text.strip()
        except Exception as e:
            if 'PerDay' in str(e) or 'per day' in str(e):
                raise RuntimeError('DAILY LIMIT reached for ' + MODEL + ' - run this cell again tomorrow')
            print('   busy, waiting...')
            time.sleep(20 * (attempt + 1))
    return None

for i, f in enumerate(tl['image'], 1):
    if f in cache:
        continue
    try:
        ans = ask(Image.open(os.path.join(LINES, f)).convert('RGB'))
    except RuntimeError as e:
        print(e); break
    if ans is not None:
        cache[f] = ans
        json.dump(cache, open(CACHE, 'w'), ensure_ascii=False, indent=1)
    print(f'{i}/{len(tl)}  {f}  ->  {ans}')
    time.sleep(13)

# --- character error rate (CER) ---
def norm(s):
    return re.sub(r'\s+', ' ', str(s)).strip()

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

tl['gemini'] = [cache.get(f, '') for f in tl['image']]
done = tl[tl['gemini'] != ''].copy()
done['errors'] = [edit_distance(norm(p), norm(t)) for p, t in zip(done['gemini'], done['text'])]
done['chars']  = [len(norm(t)) for t in done['text']]
done['cer %']  = (100 * done['errors'] / done['chars']).round(1)
done['exact']  = [int(norm(p) == norm(t)) for p, t in zip(done['gemini'], done['text'])]
done.to_csv(os.path.join(BASE, 'armenian_results.csv'), index=False)

print(f"\nLines scored: {len(done)}")
print(f"Overall CER (all errors / all characters): {100 * done['errors'].sum() / done['chars'].sum():.1f}%")
print(f"Lines read exactly right: {100 * done['exact'].mean():.1f}%")
print('\n5 worst lines:')
for _, r in done.sort_values('cer %', ascending=False).head(5).iterrows():
    print(f"{r['image']}  CER {r['cer %']}%\n   you:    {r['text']}\n   gemini: {r['gemini']}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
30 lines with your text, 30 images in text_lines
1/30  page_005_t01.png  ->  => տարամետ է
2/30  page_005_t02.png  ->  Խնդիր 2103
3/30  page_005_t03.png  ->  $\Rightarrow$ ցուցամետ է
4/30  page_005_t04.png  ->  zույգասեր է
5/30  page_006_t01.png  ->  Պար. p < 1 և q < 1 դեպքում զուգամետ է
6/30  page_006_t02.png  ->  x = 3 շրջման կետ
7/30  page_008_t01.png  ->  ...սահմանափակվում է (•Σ-ով), ինչը
8/30  page_008_t02.png  ->  է, որ Դարբուի պայմանը տեղի ունի
9/30  page_008_t03.png  ->  ֆունկցիան բնութագրիչ է.
10/30  page_008_t04.png  ->  Կրկնակի ինտեգրալի բերումը
11/30  page_008_t05.png  ->  հաջորդական ինտերվալների
12/30  page_008_t06.png  ->  ԹԵՈՐԵՄ։ Դից. $f(x,y)$ որոշված է $D=[a,b] \times$
13/30  page_008_t07.png  ->  X[c,d] ուղանկյուն մեջ, նաև սահմանա-
14/30  page_008_t08.png  ->  փակ է և ինտեգրելի է $\Delta$-ում
15/30  page_008_t09.png  ->  1. Նail, np $\forall x